<a href="https://colab.research.google.com/github/robertbarcik/ADK-tutorial/blob/main/notebooks/05_workflow_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 05 — Workflow Agents

> **⚡ Quick path** — this is one of four modules on the 1-hour course preview.
> See the [README's Quick path section](../README.md#-quick-path---1-hour) for the sequence.

Four modules in, every demo has had **one agent** doing the work. That's enough for toy problems. Real work needs composition.

ADK ships **three first-class workflow primitives** that let you compose agents into pipelines, fan-outs, and refinement loops — without writing orchestration code yourself:

- **`SequentialAgent`** — run children in order, passing state between them. A shell pipeline.
- **`ParallelAgent`** — run children concurrently, each writing to its own state key. An `asyncio.gather()`.
- **`LoopAgent`** — run children in a cycle until one of them calls `exit_loop` or the iteration limit is reached. A `while` loop with an escape hatch.

This is the single strongest pedagogical differentiator ADK has over LangGraph and CrewAI. LangGraph makes you draw your control flow as a node-and-edge graph; CrewAI hides it behind a role-playing DSL. ADK lets you name the control flow directly: *Sequential*, *Parallel*, *Loop*. Three Python classes, no graph diagrams needed.

**What you'll build:**
- A two-step pipeline: a summarizer feeds a translator.
- A three-way fan-out: researchers reporting on three countries in parallel.
- **The wow demo:** a Generator+Critic `LoopAgent` that refines a tagline until the critic stops complaining.

**Running cost:** under $0.01 across all three demos.

# Setup

In [1]:
!pip install -q google-adk==2.4.0 litellm==1.85.7 python-dotenv==1.0.1 nest-asyncio==1.6.0 deprecated==1.2.18 2>/dev/null

print("✅ Packages installed.")

✅ Packages installed.


## API Key

In [2]:
import os
OPENROUTER_API_KEY = None
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
    print("✅ API key loaded from Colab secrets.")
except Exception:
    try:
        from dotenv import load_dotenv; load_dotenv()
        OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
        if OPENROUTER_API_KEY:
            print("✅ API key loaded from .env file.")
    except ImportError:
        pass
if not OPENROUTER_API_KEY:
    from getpass import getpass
    OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")
assert OPENROUTER_API_KEY, "❌ No API key provided."
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
MODEL_STRING = "openrouter/google/gemini-2.5-flash-lite"
print(f"✅ Model: {MODEL_STRING}")

✅ API key loaded from .env file.
✅ Model: openrouter/google/gemini-2.5-flash-lite


## Imports

In [3]:
import sys, warnings, asyncio, time, uuid, logging
warnings.filterwarnings("ignore")
try:
    sys.stderr.fileno()
except Exception:
    sys.stderr = open(os.devnull, "w")
import nest_asyncio; nest_asyncio.apply()
import litellm; litellm.suppress_debug_info = True
logging.getLogger("LiteLLM").setLevel(logging.WARNING)

from google.adk.agents import LlmAgent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.models.lite_llm import LiteLlm
from google.adk.tools import exit_loop
from google.genai import types

print("✅ Imports successful.")

09:18:26 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'


09:18:26 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


✅ Imports successful.


# The Three Workflow Types — A Picture

```
SequentialAgent           ParallelAgent             LoopAgent
─────────────────         ─────────────────         ─────────────────
                                                    ┌────────────────┐
    child 1                   child 1               │   child 1      │
       │                         │                  │      │         │
       ▼                         ▼                  │      ▼         │
    child 2           ──>    child 2      <──       │   child 2      │
       │                         │                  │      │         │
       ▼                         ▼                  │      ▼ (loop)  │
    child 3                   child 3               │   child 1 ...  │
                                                    └────────────────┘
    ordered                 concurrent              until exit_loop
    pipeline                fan-out                 or max_iterations
```

Same `Runner`, same event stream, same state dict — only the composition class changes. State flows through the same mechanism: `output_key=` on each child writes its result to the session state, and subsequent children read it via the `{key}` template syntax in their instruction.

# SequentialAgent — A Two-Step Pipeline

Runs children one after another. The first child writes to state; the next child reads it; and so on. Think of it as a shell pipeline where state keys are the pipe.

Below: a summarizer (writes to `state["summary"]`) feeds a translator (reads `{summary}` from its instruction, translates to Slovak, writes to `state["translation"]`).

In [4]:
APP = "m05_demos"
USER = "student"
MODEL = LiteLlm(model=MODEL_STRING)
ss = InMemorySessionService()

summarizer = LlmAgent(
    name="summarizer",
    model=MODEL,
    description="Summarizes the user input in one sentence.",
    instruction="Read the user input. Summarize it in one clear sentence. Output only the summary.",
    output_key="summary",
)

translator = LlmAgent(
    name="translator",
    model=MODEL,
    description="Translates English to Slovak.",
    # {summary} is auto-substituted from session state before the model sees the prompt.
    instruction="Translate this English sentence to Slovak: {summary}. Output only the translation.",
    output_key="translation",
)

pipeline = SequentialAgent(
    name="summarize_then_translate",
    sub_agents=[summarizer, translator],
)

async def run(agent, prompt: str, session_id_prefix: str):
    sid = f"{session_id_prefix}-{uuid.uuid4().hex[:6]}"
    await ss.create_session(app_name=APP, user_id=USER, session_id=sid)
    runner = Runner(agent=agent, app_name=APP, session_service=ss)
    msg = types.Content(role="user", parts=[types.Part(text=prompt)])
    print(f"USER: {prompt}\n")
    async for ev in runner.run_async(user_id=USER, session_id=sid, new_message=msg):
        if ev.content and ev.content.parts:
            for p in ev.content.parts:
                if p.text and p.text.strip():
                    print(f"[{ev.author}] {p.text.strip()[:250]}")
                if p.function_call:
                    args = dict(p.function_call.args) if p.function_call.args else {}
                    print(f"[tool_call] {p.function_call.name}({args})")
    s = await ss.get_session(app_name=APP, user_id=USER, session_id=sid)
    return dict(s.state)

state = await run(pipeline, "The cat sat on the mat. It was hungry. It meowed loudly until fed.", "seq")
print("\n── Final state ──")
for k, v in state.items():
    print(f"  {k!r}: {v!r}")

USER: The cat sat on the mat. It was hungry. It meowed loudly until fed.



[summarizer] The hungry cat sat on the mat and meowed loudly until it was fed.


[translator] Hladná mačka sedela na rohožke a nahlas mňaukala, kým ju nenakŕmili.

── Final state ──
  'summary': 'The hungry cat sat on the mat and meowed loudly until it was fed.'
  'translation': 'Hladná mačka sedela na rohožke a nahlas mňaukala, kým ju nenakŕmili.'


Two agents ran in order. The summarizer wrote to `state["summary"]`. The translator's instruction contained `{summary}` — ADK substituted the value from state before the model saw the prompt, so the translator worked on the summary, not the original.

The `{key}` template is the glue that makes workflow agents composable. Write to state with `output_key=`, read from state with `{key}` in a later agent's instruction. No orchestration code.

# ParallelAgent — Fan-Out, Concurrent

Runs all children concurrently (via `asyncio`), each writing to its own state key. Total wall time ≈ slowest child, not sum of children.

Below: three researchers report one fact each about three different countries. All three run at once.

In [5]:
de_researcher = LlmAgent(
    name="de_researcher", model=MODEL,
    instruction="State one interesting fun fact about Germany in one sentence.",
    output_key="germany_fact",
)
sk_researcher = LlmAgent(
    name="sk_researcher", model=MODEL,
    instruction="State one interesting fun fact about Slovakia in one sentence.",
    output_key="slovakia_fact",
)
cz_researcher = LlmAgent(
    name="cz_researcher", model=MODEL,
    instruction="State one interesting fun fact about Czech Republic in one sentence.",
    output_key="czech_fact",
)

trio = ParallelAgent(
    name="trio",
    sub_agents=[de_researcher, sk_researcher, cz_researcher],
)

# Time the parallel run
t0 = time.time()
state = await run(trio, "Give me three country facts.", "par")
print(f"\n⏱  Total wall time: {time.time() - t0:.2f}s")
print("\n── Final state ──")
for k, v in state.items():
    if k.endswith("_fact"):
        print(f"  {k!r}: {v}")

USER: Give me three country facts.



[sk_researcher] Here are three facts about Slovakia:

1.  Slovakia is home to over 6,000 caves, with the Ochtinská Aragonite Cave being a UNESCO World Heritage site known for its unique aragonite formations.
2.  The Spiš Castle complex in eastern Slovakia is one of 
[de_researcher] Certainly! Here are three fun facts about Germany:

1.  Germany has over 20,000 castles, a higher density than almost any other country in the world.
2.  It's illegal to run out of gas on the Autobahn, as it's considered a traffic obstruction and can
[cz_researcher] Here are three fun facts about the Czech Republic:

1.  The Czech Republic has the highest beer consumption per capita in the world.
2.  The inventor of soft contact lenses, Otto Wichterle, was Czech, and he famously developed his initial prototypes 

⏱  Total wall time: 0.90s

── Final state ──
  'slovakia_fact': Here are three facts about Slovakia:

1.  Slovakia is home to over 6,000 caves, with the Ochtinská Aragonite Cave being a UNESCO World

Notice the authors in the event stream — `de_researcher`, `sk_researcher`, `cz_researcher` interleave in whatever order they finish, not the order you declared them. That's concurrency.

And the wall time is roughly one child's duration, not three. If each LLM call is ~1s, a sequential run would take ~3s; a parallel run takes ~1.1s. The savings scale with the width of the fan-out.

A useful pattern for production: fan out independent research / lookups / API calls in parallel, then feed the collected state into a SequentialAgent that synthesizes a final answer. We'll compose exactly that in a minute.

# LoopAgent — The Generator+Critic Wow Demo

The canonical ADK composition. Two agents cycle: a **generator** produces a draft, a **critic** evaluates it. If the critic is not satisfied, it writes a critique back to state and the loop continues — the generator reads the critique on the next iteration and revises. When the critic is satisfied, it calls the `exit_loop` tool and the loop terminates.

This pattern goes by many names — *critic-driven refinement*, *self-correction*, *reflexion*. In plain agent frameworks you'd write it yourself with a `while` loop. In ADK, it's a `LoopAgent` with two children and a `max_iterations=` guard.

In [6]:
generator = LlmAgent(
    name="generator",
    model=MODEL,
    description="Writes or revises a tagline.",
    instruction="""Write or revise a 1-2 sentence tagline for a course on Google's ADK,
aimed at software engineers. Requirements: specific, concrete, avoids marketing cliches
("unlock", "unleash", "empower", "transform").

If a previous draft and critique exist, revise the draft to address the critique.
Otherwise, write a fresh first draft.

Previous draft: {draft?}
Previous critique: {critique?}

Output ONLY the new tagline. No preamble, no markdown.""",
    output_key="draft",
)

critic = LlmAgent(
    name="critic",
    model=MODEL,
    description="Critiques drafts; calls exit_loop when satisfied.",
    instruction="""You critique taglines. Read this draft:

Draft: {draft}

Evaluate against these criteria:
- Does it say something SPECIFIC about the course content?
- Does it avoid marketing cliches ("unlock", "unleash", "empower", "transform", "elevate", "master")?
- Is it under 2 sentences?

If the draft FAILS any criterion, write a one-sentence critique explaining which criterion failed and how.

If the draft PASSES all three criteria, call the exit_loop tool. Do NOT output text in that case.""",
    output_key="critique",
    tools=[exit_loop],
)

refiner = LoopAgent(
    name="tagline_refiner",
    sub_agents=[generator, critic],
    max_iterations=5,
)

state = await run(refiner, "Write a tagline.", "loop")
print("\n── Final draft ──")
print(state.get("draft", "(none)"))

USER: Write a tagline.



[generator] Master Google's ADK to build custom Android device experiences. Integrate hardware and software with advanced development techniques.


[critic] The tagline uses the marketing cliche "Master".


[generator] Develop custom Android device experiences using Google's ADK. Integrate hardware and software with advanced development techniques.


[tool_call] exit_loop({})

── Final draft ──
Develop custom Android device experiences using Google's ADK. Integrate hardware and software with advanced development techniques.


Walk through what you just watched.

- **Iteration 1:** generator wrote a first draft. No previous critique to address.
- **Critic 1:** read the draft, saw a cliche or vagueness, wrote a critique.
- **Iteration 2:** generator read the critique via `{critique?}`, revised the draft to address it.
- **Critic 2:** maybe still not satisfied. New critique. Loop continues.
- **Eventually:** critic found all three criteria satisfied, called `exit_loop`. The loop terminated.

Two things to notice in the code:

- `{draft?}` and `{critique?}` use the `?` suffix — "optional state key, do not error if missing." The generator runs before the critic on iteration 1 when neither key exists yet.
- `max_iterations=5` is the safety net. If the critic is impossible to satisfy, the loop stops after 5 cycles. Always set this; never rely on the critic to always exit.

This is the pattern you'd otherwise implement as:

```python
# Without LoopAgent, you'd write something like this yourself:
state = {"draft": None, "critique": None}
for i in range(5):
    state["draft"] = await generator.run(state)
    state["critique"] = await critic.run(state)
    if critic_is_satisfied(state["critique"]):
        break
```

The LoopAgent version is equivalent, plus: events for every step, state history for auditing, `max_iterations` as declaration not runtime-code, and the `exit_loop` tool as an explicit signal the critic is satisfied. Worth the 20 lines of saved orchestration code.

# Composing Workflows

Workflow agents are themselves agents. You can nest them. A `SequentialAgent` can contain a `ParallelAgent` as one of its children; a `LoopAgent` can contain a `SequentialAgent`; and so on.

A concrete pattern for production agents: fan out independent research in parallel, then synthesize sequentially.

In [7]:
# Research three countries in parallel, then synthesize.
synthesizer = LlmAgent(
    name="synthesizer",
    model=MODEL,
    description="Writes a short combined report from three facts.",
    instruction="""You have three country facts in session state. Write a short
3-sentence combined report that connects them thematically if possible.

Germany: {germany_fact}
Slovakia: {slovakia_fact}
Czech Republic: {czech_fact}

Output only the report.""",
    output_key="report",
)

pipeline2 = SequentialAgent(
    name="research_pipeline",
    sub_agents=[
        trio,           # ← the ParallelAgent from earlier
        synthesizer,
    ],
)

t0 = time.time()
state = await run(pipeline2, "Research and synthesize.", "compose")
print(f"\n⏱  Total wall time: {time.time() - t0:.2f}s")
print("\n── Final report ──")
print(state.get("report", ""))

USER: Research and synthesize.



[cz_researcher] The Czech Republic has one of the highest beer consumption rates per capita in the world, with Czechs famously drinking more beer than any other nation.
[de_researcher] Germany has over 1,500 different types of beer, and the Reinheitsgebot (Bavarian Purity Law) of 1516 is still largely followed today.


[sk_researcher] Slovakia is home to the world's largest total number of castles and chateaux per capita, boasting over 180 castles and 500 chateaux.


[synthesizer] Central Europe offers a rich tapestry of cultural heritage, evident in its historical landmarks and beloved beverages. Slovakia boasts an impressive density of castles and chateaux, reflecting a storied past. Meanwhile, Germany upholds its beer purit

⏱  Total wall time: 1.40s

── Final report ──
Central Europe offers a rich tapestry of cultural heritage, evident in its historical landmarks and beloved beverages. Slovakia boasts an impressive density of castles and chateaux, reflecting a storied past. Meanwhile, Germany upholds its beer purity laws with over 1,500 varieties, a tradition mirrored in the Czech Republic's world-leading per capita beer consumption.


Five LLM calls — three researchers in parallel, then a synthesizer — and the wall time is close to two calls' worth, not five. Production agents are built out of compositions like this.

# When to Use a Workflow Agent vs. LLM-Driven Flow

The big alternative to workflow agents is: let the LLM decide.

```python
# LLM-driven: the orchestrator's LLM picks which tool/sub-agent to call next.
orchestrator = LlmAgent(
    name="orchestrator",
    sub_agents=[summarizer, translator, researcher_de, ...],
    instruction="Figure out what to do based on user input.",
)
```

Both styles work. They're not interchangeable.

| Use a **workflow agent** when... | Use **LLM-driven** when... |
|---|---|
| The control flow is fixed — always summarize, then translate | The control flow depends on user input |
| You need determinism (tests, evals) | You need flexibility |
| Latency matters — Parallel fans out without an LLM turn of deliberation | The model's judgment is the point |
| You want the workflow to be auditable from a diagram | Conversations can go anywhere |

Rule of thumb: **if you can name the workflow, use a workflow agent. If you can't, let the LLM decide.**

Sequential, Parallel, Loop cover the named-workflow cases cleanly. Anything more complex, you start composing them. And if the composition gets unwieldy, that's a sign the workflow shouldn't be named after all — let the LLM drive.

# Your Turn

1. **A three-step pipeline.** Add a third step after `summarizer → translator` that takes the translation and writes a haiku about it (`output_key="haiku"`). Run the pipeline. Do all three results appear in state?
2. **Parallel then sequential.** Build a sequential pipeline whose first step is the `trio` parallel agent and whose second step writes a headline summarizing all three facts. (Hint: you already have `synthesizer` — swap its instruction for a headline.)
3. **A stricter critic.** In the generator+critic loop, modify the critic to also reject taglines over 20 words. Run the loop. Does it ever converge, or does it max out at `max_iterations`?
4. **A loop with one child.** LoopAgent works with a single child too. Build a `LoopAgent(name="persistent_greeter", sub_agents=[greeter], max_iterations=3)` and see what happens — does the same greeter run three times?

# Key Takeaways

- **Three workflow primitives:** `SequentialAgent` (ordered pipeline), `ParallelAgent` (concurrent fan-out), `LoopAgent` (cycle until `exit_loop` or `max_iterations`).
- **State is the pipe:** `output_key="foo"` writes to `state["foo"]`; `{foo}` in a later agent's instruction reads it. `{foo?}` is "optional, don't error if missing."
- **Always set `max_iterations`** on a LoopAgent. The critic might be impossible to satisfy.
- **`exit_loop`** is the tool the LoopAgent's child calls to stop. Give it to whichever child has the exit condition.
- **Compose workflows:** Parallel inside Sequential is the classic "fan out research, then synthesize" pattern. Production agents are built out of compositions.
- **Rule:** if you can name the workflow, use a workflow agent. If you can't, let the LLM decide.

# Next up — M06: Multi-agent hierarchies

Workflow agents give you control flow by declaration. M06 gives you the other half — **LLM-driven routing** between agents. `sub_agents` for transfer, `AgentTool` for consultant-style calls (we saw a glimpse of `AgentTool` in M02). Same agent, two coordination patterns, different trade-offs. See you there.